In [2]:
from langgraph.graph import StateGraph,END,START

from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv
import os



In [5]:
load_dotenv(dotenv_path='.env',override=True)

True

In [8]:
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

In [9]:
model = ChatGroq(
    model="llama-3.3-70b-versatile",  
    api_key=GROQ_API_KEY  
)

In [10]:
class query(TypedDict):
    title:str
    outline:str
    blog:str

In [11]:
graph=StateGraph(query)

In [12]:
def outline(state:query) -> query:
    title=state['title']
    prompt=f'create a detailed outline for blog on the title {title}'
    outline=model.invoke(prompt).content
    state['outline']=outline
    return state

In [13]:
def blog(state:query) -> query:
    title=state['title']
    outline=state['outline']
    prompt=f'create a detailed note on title {title} by following these outlines {outline}'
    blog=model.invoke(prompt).content
    state['blog']=blog
    return state

In [14]:
graph.add_node('outline',outline)
graph.add_node('blog',blog)

In [15]:
graph.add_edge(START,'outline')
graph.add_edge('outline','blog')
graph.add_edge('blog',END)

In [16]:
workflow=graph.compile()

In [17]:
initial_state={'query':'AI'}
final_state=workflow.invoke(initial_state)
print(final_state['blog'].content)


KeyError: 'title'